<a href="https://colab.research.google.com/github/ABRohanRoy/CodingStreak/blob/main/DogsvsCats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [4]:
!kaggle datasets download -d salader/dogs-vs-cats

Dataset URL: https://www.kaggle.com/datasets/salader/dogs-vs-cats
License(s): unknown
 99% 1.05G/1.06G [00:04<00:00, 242MB/s]
100% 1.06G/1.06G [00:04<00:00, 262MB/s]


In [7]:
import zipfile
zip_ref=zipfile.ZipFile('/content/dogs-vs-cats.zip','r')
zip_ref.extractall('/content')
zip_ref.close()

In [10]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Conv2D,MaxPool2D,Flatten,BatchNormalization,Dropout

In [11]:
train_ds=keras.utils.image_dataset_from_directory(
    directory='/content/train',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256,256)
)

Found 20000 files belonging to 2 classes.


In [12]:
validation_ds=keras.utils.image_dataset_from_directory(
    directory='/content/test',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256,256)
)

Found 5000 files belonging to 2 classes.


In [13]:
# Train_ds =train_ds/255
#validation_ds=validation_ds/255
# NOrmalize

def process(image,label):
  image=tf.cast(image/255. ,tf.float32)
  return label

In [14]:
model = Sequential()
model.add(Conv2D(32,kernel_size=(3,3),padding='valid',activation='relu',input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(64,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(128,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(64,activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(1,activation='sigmoid'))
model.add(Dropout(0.1))


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [15]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [16]:
model.fit(train_ds,epochs=10,validation_data=validation_ds)

Epoch 1/10
315/625 ━━━━━━━━━━━━━━━━━━━━ 38:47 8s/step - accuracy: 0.5208 - loss: 7.4564

KeyboardInterrupt: 

In [ ]:
import pickle
pickle.dump(model,open('model.pkl','wb'))

In [19]:
from tensorflow.keras.applications import VGG16
model =VGG16(weights='imagenet',input_shape=(224,224,3))

553467096/553467096 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [21]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
import numpy as np


In [22]:
model = VGG16(weights='imagenet', input_shape=(224, 224, 3))


In [23]:
img_path = 'dog.jpg'

# Load and resize image to 224x224 as required by VGG16
img = image.load_img(img_path, target_size=(224, 224))

# Convert image to numpy array
img_array = image.img_to_array(img)

# Expand dimensions to match model input shape (1, 224, 224, 3)
img_array = np.expand_dims(img_array, axis=0)

# Preprocess input for VGG16
img_array = preprocess_input(img_array)


In [24]:
preds = model.predict(img_array)


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


In [25]:
print('Predicted:', decode_predictions(preds, top=3)[0])


35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Predicted: [('n02113023', 'Pembroke', np.float32(0.20441712)), ('n02109961', 'Eskimo_dog', np.float32(0.111628294)), ('n02106030', 'collie', np.float32(0.10154232))]
